In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

In [5]:
df = pd.read_csv("../data/Snapshot_2026/Snapshot_2026_W14_Clean.csv")
print(df.head())

  Snapshot_Date MRP_ELEMENT_DATA  DELIVERY_SCHEDULE_LINE_COUNTER  \
0    2026-03-31              NaN                             NaN   
1    2026-03-31              NaN                             NaN   
2    2026-03-31              NaN                             NaN   
3    2026-03-31              NaN                             NaN   
4    2026-03-31              NaN                             NaN   

   MATERIAL_NUMBER MRP_TYPE  SAFETY_TIME  GOODS_RECEIPT_PROCESSING_TIME  \
0       6995225001       YS            0                              3   
1       7238207001       YS            0                              4   
2       6918689001       YS            0                              2   
3       8782245001       YS            0                              1   
4       5445990001       YS            7                              1   

   SAFETY_STOCK JOINT_COMMODITY_CODE  PLANT  ...    SOURCING_MANAGER Delay  \
0            19            Y91150106   4425  ...  HD6 - Sanjay

/var/folders/mx/2d9qj2l15sd94vd52vrjrkb80000gn/T/ipykernel_91691/3503851238.py:1: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Snapshot_2026/Snapshot_2026_W14_Clean.csv")


In [9]:
print("Nombre de valeurs manquantes dans 'comment' :", 
df["comment"].isna().sum())

Nombre de valeurs manquantes dans 'comment' : 66153


In [10]:
df["comment"] = df["comment"].fillna("")

In [11]:
vectorizer = TfidfVectorizer(max_features=100)
X_comments = vectorizer.fit_transform(df["comment"])

In [14]:
# %% [Cellule 3: Prétraitement]
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Vérifie les valeurs manquantes dans "Comment"
print("Nombre de valeurs manquantes dans 'comment' :", df["comment"].isna().sum())

# 2. Supprime les lignes avec des commentaires vides
df = df.dropna(subset=["comment"])

# 3. Nettoie les commentaires (supprime les caractères spéciaux, met en minuscules)
df["Cleaned_Comment"] = df["comment"].str.lower().str.replace(r'[^\w\s]', '', regex=True)

# 4. Vectorise les commentaires avec TF-IDF
vectorizer = TfidfVectorizer(max_features=100, stop_words=['the', 'a', 'an', 'and', 'or', 'der', 'die', 'das'])
X_comments = vectorizer.fit_transform(df["Cleaned_Comment"])

# 5. Affiche la forme de la matrice TF-IDF
print("Forme de la matrice TF-IDF :", X_comments.shape)

Nombre de valeurs manquantes dans 'comment' : 0
Forme de la matrice TF-IDF : (68140, 100)


In [20]:
from sklearn.preprocessing import LabelEncoder

# 1. Initialiser les LabelEncoders
le_supplier = LabelEncoder()
le_material = LabelEncoder()
le_category = LabelEncoder()

# 2. Encoder les colonnes
df["Supplier_Encoded"] = le_supplier.fit_transform(df["VENDOR_ACCOUNT_NUMBER"])
df["Material_Encoded"] = le_material.fit_transform(df["MATERIAL_NUMBER"])
df["Category_Encoded"] = le_category.fit_transform(df["MATERIAL_STATUS"])

# 3. Afficher un exemple des encodages
print("Encodage des fournisseurs :", dict(zip(le_supplier.classes_, le_supplier.transform(le_supplier.classes_))))
print("Encodage des matériaux :", dict(zip(le_material.classes_, le_material.transform(le_material.classes_))))

Encodage des fournisseurs : {600900: 0, 616400: 1, 10000617: 2, 10000827: 3, 10001937: 4, 10002104: 5, 10002627: 6, 10002731: 7, 10003703: 8, 10003975: 9, 10005556: 10, 10005807: 11, 10006309: 12, 10010014: 13, 10016321: 14, 10017251: 15, 10017422: 16, 10017496: 17, 10019173: 18, 10019618: 19, 10020582: 20, 10021128: 21, 10023196: 22, 10023758: 23, 10029946: 24, 10030584: 25, 10030892: 26, 10037265: 27, 10037420: 28, 10037712: 29, 10037959: 30, 10038280: 31, 10044788: 32, 10044942: 33, 10045256: 34, 10052036: 35, 10058994: 36, 10059085: 37, 10059206: 38, 10059223: 39, 10059339: 40, 10060087: 41, 10066782: 42, 10073452: 43, 10074089: 44, 10074100: 45, 10074643: 46, 10077177: 47, 10084346: 48, 10100567: 49, 10100968: 50, 10101802: 51, 10101865: 52, 10101970: 53, 10102110: 54, 10102123: 55, 10102147: 56, 10102549: 57, 10102883: 58, 10104541: 59, 10104818: 60, 10106571: 61, 10108478: 62, 10108519: 63, 10108524: 64, 10108538: 65, 10108541: 66, 10108585: 67, 10108624: 68, 10108644: 69, 10108

In [21]:
import numpy as np

# 1. Créer la cible : 1 si la prochaine catégorie est "Stock Out", 0 sinon
df = df.sort_values(["Material_ID", "Timestamp"])  # Trier par matériau et date
df["Next_Category"] = df.groupby("Material_ID")["Category"].shift(-1)  # Catégorie suivante
df["Target"] = (df["Next_Category"] == "Stock Out").astype(int)  # 1 si Stock Out, 0 sinon
df = df.dropna(subset=["Target"])  # Supprimer les lignes sans "Next_Category"

# 2. Construire la matrice de features (X)
X = np.hstack([
    df[["Supplier_Encoded", "Material_Encoded", "Category_Encoded"]].values,  # Features catégorielles
    X_comments.toarray()  # Features TF-IDF des commentaires
])
y = df["Target"]  # Cible (0 ou 1)

# 3. Afficher les dimensions
print("Forme de X :", X.shape)
print("Forme de y :", y.shape)

KeyError: 'Material_ID'

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Séparer en train/test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Garde la même proportion de 0/1 dans train et test
)

# 2. Afficher les tailles
print("Taille de X_train :", X_train.shape)
print("Taille de X_test :", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialiser et entraîner le modèle
model = RandomForestClassifier(
    n_estimators=100,  # Nombre d'arbres
    random_state=42,   # Pour la reproductibilité
    class_weight="balanced"  # Équilibre les classes (0 et 1)
)
model.fit(X_train, y_train)

# 2. Prédire sur l'ensemble de test
y_pred = model.predict(X_test)

# 3. Évaluer le modèle
print("Accuracy :", accuracy_score(y_test, y_pred))
print("\nClassification Report :\n", classification_report(y_test, y_pred))

In [ ]:
# %% [Cellule 8: Sauvegarde du modèle]
import joblib
import os

# 1. Créer le dossier de sortie si nécessaire
os.makedirs("outputs/predictions", exist_ok=True)

# 2. Sauvegarder le modèle
joblib.dump(model, "outputs/predictions/risk_prediction_model.pkl")

# 3. Sauvegarder le vectorizer TF-IDF
joblib.dump(vectorizer, "outputs/predictions/tfidf_vectorizer.pkl")

# 4. Sauvegarder les LabelEncoders
joblib.dump({
    "Supplier": le_supplier,
    "Material": le_material,
    "Category": le_category
}, "outputs/predictions/label_encoders.pkl")

print("✅ Modèle, vectorizer et encoders sauvegardés dans outputs/predictions/")

In [ ]:
# %% [Cellule 9: Test manuel du modèle]
# 1. Charger le modèle et les outils
model = joblib.load("outputs/predictions/risk_prediction_model.pkl")
vectorizer = joblib.load("outputs/predictions/tfidf_vectorizer.pkl")
label_encoders = joblib.load("outputs/predictions/label_encoders.pkl")

# 2. Créer un nouveau commentaire de test
new_comment = "Urgent: Supplier X delay for MAT_100"
new_supplier = "Supplier_X"
new_material = "MAT_100"
new_category = "Potential Stock Out"

# 3. Prétraiter le nouveau commentaire
new_comment_cleaned = new_comment.lower().replace(r'[^\w\s]', '', regex=True)
new_comment_vectorized = vectorizer.transform([new_comment_cleaned])

# 4. Encoder les variables catégorielles
new_supplier_encoded = label_encoders["Supplier"].transform([new_supplier])[0]
new_material_encoded = label_encoders["Material"].transform([new_material])[0]
new_category_encoded = label_encoders["Category"].transform([new_category])[0]

# 5. Construire les features
X_new = np.hstack([
    np.array([[new_supplier_encoded, new_material_encoded, new_category_encoded]]),
    new_comment_vectorized.toarray()
])

# 6. Prédire
prediction = model.predict(X_new)[0]
probability = model.predict_proba(X_new)[0][1]  # Probabilité de "Stock Out"

print(f"Commentaire : {new_comment}")
print(f"Prédiction : {'Stock Out la semaine prochaine' if prediction else 'Pas de Stock Out'}")
print(f"Probabilité : {probability*100:.1f}%")